In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import catboost as cb
from sklearn.model_selection import KFold, StratifiedKFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings("ignore")
#Config
TARGET = 'Y'
ID_COL = 'X1'
N_FOLDS = 10
SEED = 42
LOG_TARGET = False  # Set to True if the target is skewed
N_ESTIMATORS = 10000
EARLY_STOPPING_ROUNDS = 300
VERBOSE = 500

In [ ]:

# ==================== LOAD DATA ====================
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

train_orig = train.copy()
test_orig = test.copy()

y = train[TARGET].copy()
if LOG_TARGET:
    y = np.log1p(y)

train = train.drop(columns=[TARGET])


## Feature Engineering

In [ ]:
all_data = pd.concat([train, test], axis=0, sort=False).reset_index(drop=True)

# 1. Store missingness information as a feature
for col in all_data.columns:
    if col not in [ID_COL]:
        all_data[col + '_was_missing'] = all_data[col].isnull().astype(int)

# 2. Identify column types
cat_cols = all_data.select_dtypes(include=['object']).columns.tolist()
num_cols = all_data.select_dtypes(include=['int64', 'float64']).columns.tolist()
num_cols = [c for c in num_cols if c not in [ID_COL] and not c.endswith('_was_missing')]

print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")

# 3. Handle missing values in numerical columns (fill with median)
for col in num_cols:
    median_val = all_data[col].median()
    all_data[col].fillna(median_val, inplace=True)

# 4. Label Encode categorical columns for easier feature engineering
label_encoders = {}
for col in cat_cols:
    all_data[col] = all_data[col].astype(str)
    le = LabelEncoder()
    all_data[col] = le.fit_transform(all_data[col])
    label_encoders[col] = le

# 5a. Interaction between numerical features
if 'X2' in num_cols and 'X6' in num_cols:
    all_data['X2_div_X6'] = all_data['X2'] / (all_data['X6'] + 1e-5)
    all_data['X2_times_X6'] = all_data['X2'] * all_data['X6']

if 'X4' in num_cols and 'X6' in num_cols:
    all_data['X4_div_X6'] = all_data['X4'] / (all_data['X6'] + 1e-5)

# 5b. Aggregations (Group Statistics)
for col in cat_cols[:2]: 
    for col2 in cat_cols[1:3]:
        all_data[f'cat_{col}_plus_{col2}'] = all_data[col].astype(str) + "_" + all_data[col2].astype(str)

new_cat_cols = [c for c in all_data.columns if c.startswith('cat_')]
for col in new_cat_cols:
    le = LabelEncoder()
    all_data[col] = le.fit_transform(all_data[col].astype(str))
    cat_cols.append(col)  

print(f"Created new categorical features: {new_cat_cols}")

train = all_data[all_data.index < len(train_orig)].reset_index(drop=True)
test = all_data[all_data.index >= len(train_orig)].reset_index(drop=True)

for col in cat_cols:
    train[col] = train[col].astype('category')
    test[col] = test[col].astype('category')

print("\nFinal shape of train:", train.shape)
print("Final shape of test:", test.shape)


Categorical columns: ['X1', 'X3', 'X5', 'X7', 'X9', 'X10', 'X11']
Numerical columns: ['X2', 'X4', 'X6', 'X8']

--- Creating New Features ---
Created new categorical features: ['cat_X1_plus_X3', 'cat_X1_plus_X5', 'cat_X3_plus_X3', 'cat_X3_plus_X5']

Final shape of train: (6000, 28)
Final shape of test: (2523, 28)


In [4]:
num_bins = 10
y_for_bins = pd.qcut(y, q=num_bins, labels=False, duplicates='drop')
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

In [ ]:
oof_lgb = np.zeros(len(train))
oof_cat = np.zeros(len(train))

# Arrays to store test predictions (from each fold)
test_lgb = np.zeros(len(test))
test_cat = np.zeros(len(test))

# For meta-features, we'll store the OOF predictions as features for the stacker
meta_features_train = np.zeros((len(train), 2))
meta_features_test = np.zeros((len(test), 2))

for fold, (tr_idx, val_idx) in enumerate(skf.split(train, y_for_bins)):
    print(f'\n========== Fold {fold+1} ==========')

    X_tr, X_val = train.iloc[tr_idx], train.iloc[val_idx]
    y_tr, y_val = y.iloc[tr_idx], y.iloc[val_idx]

    # LIGHTGBM 
    
    lgb_model = lgb.LGBMRegressor(
        objective='regression',
        metric='rmse',
        boosting_type='gbdt',
        n_estimators=N_ESTIMATORS,
        learning_rate=0.01,
        num_leaves=63,
        max_depth=-1,
        min_child_samples=100,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=5.0,
        reg_lambda=5.0,
        random_state=SEED + fold,
        n_jobs=-1,
        verbose=-1
    )

    lgb_model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        categorical_feature=cat_cols,
        callbacks=[lgb.early_stopping(EARLY_STOPPING_ROUNDS), lgb.log_evaluation(VERBOSE)]
    )

    oof_lgb[val_idx] = lgb_model.predict(X_val)
    test_lgb += lgb_model.predict(test) / N_FOLDS

    cat_model = cb.CatBoostRegressor(
        loss_function='RMSE',
        learning_rate=0.03,
        depth=8,
        l2_leaf_reg=5.0,
        iterations=N_ESTIMATORS,
        random_seed=SEED + fold,
        early_stopping_rounds=EARLY_STOPPING_ROUNDS,
        verbose=VERBOSE,
        thread_count=-1,
        bootstrap_type='Bernoulli',
        subsample=0.8
    )

    cat_model.fit(
        X_tr, y_tr,
        eval_set=(X_val, y_val),
        cat_features=cat_cols,
        use_best_model=True
    )

    oof_cat[val_idx] = cat_model.predict(X_val)
    test_cat += cat_model.predict(test) / N_FOLDS



========== Fold 1 ==========
Training LightGBM...
Training until validation scores don't improve for 300 rounds
[500]	valid_0's rmse: 1065.16
Early stopping, best iteration is:
[399]	valid_0's rmse: 1064.45
Training CatBoost...
0:	learn: 1673.2471131	test: 1703.6962645	best: 1703.6962645 (0)	total: 84.1ms	remaining: 14m
500:	learn: 939.0528599	test: 1057.9547178	best: 1052.3440241 (320)	total: 23.2s	remaining: 7m 20s
Stopped by overfitting detector  (300 iterations wait)

bestTest = 1052.344024
bestIteration = 320

Shrink model to first 321 iterations.

========== Fold 2 ==========
Training LightGBM...
Training until validation scores don't improve for 300 rounds
[500]	valid_0's rmse: 1127.06
Early stopping, best iteration is:
[338]	valid_0's rmse: 1123.78
Training CatBoost...
0:	learn: 1672.4760605	test: 1704.1091488	best: 1704.1091488 (0)	total: 26.4ms	remaining: 4m 23s
500:	learn: 944.0048288	test: 1124.1808548	best: 1120.1562604 (236)	total: 20.4s	remaining: 6m 26s
Stopped by over

In [6]:
cv_score_lgb = np.sqrt(mean_squared_error(y, oof_lgb))
cv_score_cat = np.sqrt(mean_squared_error(y, oof_cat))

In [ ]:
meta_features_train[:, 0] = oof_lgb
meta_features_train[:, 1] = oof_cat
meta_features_test[:, 0] = test_lgb
meta_features_test[:, 1] = test_cat

from sklearn.linear_model import Ridge
meta_model = Ridge(alpha=1.0, random_state=SEED)
meta_model.fit(meta_features_train, y)

stacked_oof = meta_model.predict(meta_features_train)
stacked_test = meta_model.predict(meta_features_test)

stacked_cv_score = np.sqrt(mean_squared_error(y, stacked_oof))

In [ ]:
final_preds = stacked_test

if LOG_TARGET:
    final_preds = np.expm1(final_preds)
    print("Applying inverse log1p transformation to predictions.")

submission = pd.DataFrame({
    ID_COL: range(0,len(test_orig)),
    TARGET: final_preds
})


submission.to_csv("submission.csv", index=False)


In [10]:
stacked_cv_score

np.float64(1072.7406507529654)